# Classical Gram-Schmidt Loses Orthogonality

**Topic:** Numerical stability &nbsp;·&nbsp; **Difficulty:** Hard

Classical and modified Gram-Schmidt compute the same thing in exact arithmetic.
In floating point they do not: on a nearly-dependent basis, the classical version
produces vectors that are visibly not orthogonal, and the modified version (the
same operations, reordered) stays accurate.

Implement both and measure `||Q^T Q - I||`. The reordering is one line.

## The data (created in the next cell)

- `A` `(80, 12)`: columns are nearly linearly dependent by construction.
- Condition number around `1e6`.

## Your task

With **NumPy only**:

1. `classical_gs(A)`: for each column `j`, subtract its projections onto **all**
   previously computed `q` vectors at once, using the *original* column each
   time, then normalize.
2. `modified_gs(A)`: for each column `j`, subtract the projection onto `q_0`,
   then re-project the *updated* vector onto `q_1`, and so on one at a time,
   then normalize.
3. For each, compute `loss = ||Q^T Q - I||_F`.
4. Also compute the same for `np.linalg.qr`.
5. Set `answers` to a dict with `"loss_classical"`, `"loss_modified"`,
   `"loss_numpy"`, `"cond_A"`.

## Evaluation

Compared to the reference: `cond_A` to a relative tolerance of 1e-6, the classical
loss to within 50% (it shifts by a few percent from one NumPy build to the next),
and the other two losses to an absolute 1e-6. The evaluation cell also checks the
modified version is more than 100x more orthogonal than the classical one.

In [ ]:
import numpy as np

rng = np.random.default_rng(3004)
m, k = 80, 12
base = rng.normal(0, 1, (m, 1))
A = base + rng.normal(0, 1e-5, (m, k))         # columns nearly identical
A[:, 0] = base.ravel()
print(f"A {A.shape}, condition number = {np.linalg.cond(A):.2e}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("loss_classical", [], 4.997450154577707e-05, -8.367631330169032e-06, 2.5e-05),
    ("loss_modified", [], 9.216929772528422e-11, -1.5432644237968123e-11, 1e-06),
    ("loss_numpy", [], 1.6577776062572951e-15, -2.7757499139565393e-16, 1e-06),
    ("cond_A", [], 1530447.115774127, -256255.0268437958, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "loss_modified" in answers:
    _better = answers["loss_modified"] * 100 < answers["loss_classical"]
    print(f"  {'modified 100x better':<24} {'ok' if _better else 'wrong'}  "
          f"({answers['loss_modified']:.2e} vs {answers['loss_classical']:.2e})")
    _ok = _ok and _better

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet, revisit your solution and try again.")

## Solution

Classical Gram-Schmidt computes every projection coefficient from the *original*
column, so the rounding error in each `q` is never seen by the later
subtractions and the errors accumulate independently. Modified Gram-Schmidt
subtracts sequentially from the running remainder, so each step works with the
error already made and corrects for it. The operations are algebraically
identical and numerically worlds apart: the classical loss scales like
`kappa^2`, the modified like `kappa`.

```python
import numpy as np

def classical_gs(A):
    m, k = A.shape
    Q = np.zeros((m, k))
    for j in range(k):
        v = A[:, j].copy()
        if j > 0:
            coef = Q[:, :j].T @ A[:, j]        # from the ORIGINAL column
            v = v - Q[:, :j] @ coef
        Q[:, j] = v / np.linalg.norm(v)
    return Q

def modified_gs(A):
    m, k = A.shape
    V = A.copy().astype(float)
    Q = np.zeros((m, k))
    for j in range(k):
        v = V[:, j]
        for i in range(j):
            v = v - (Q[:, i] @ v) * Q[:, i]    # from the RUNNING remainder
        Q[:, j] = v / np.linalg.norm(v)
    return Q

def loss(Q):
    k = Q.shape[1]
    return float(np.linalg.norm(Q.T @ Q - np.eye(k)))

Qn, _ = np.linalg.qr(A)

answers = {
    "loss_classical": loss(classical_gs(A)),
    "loss_modified": loss(modified_gs(A)),
    "loss_numpy": loss(Qn),
    "cond_A": float(np.linalg.cond(A)),
}
```